In [0]:
%run ../connection

In [0]:
# -----------------------------------------------------------------------------
# CELL 1: IMPORTS
# Why: We only use the standard library + requests. No Spark needed here —
#      we are just making HTTP calls and writing files. Keeping it lightweight
#      means faster startup and simpler debugging.
# -----------------------------------------------------------------------------
 
import requests   # HTTP calls to CoinGecko REST API
import json       # Serialize Python dicts → JSON strings before saving
import uuid       # Generate a unique ID (UUID4) for each pipeline run
import time       # Sleep between API calls to respect rate limits
import logging    # Structured logging (better than bare print for production)
 
from datetime import datetime, timezone   # Always work in UTC — never local time
from typing   import Any, Dict, List, Optional

In [0]:
%run ./config

In [0]:
# -----------------------------------------------------------------------------
# CELL 4: HELPER FUNCTIONS
# -----------------------------------------------------------------------------
 
def call_api(endpoint: str, params: Optional[Dict] = None) -> Any:
    """
    Make a GET request to the CoinGecko API with retry logic.
 
    WHY RETRY LOGIC?
      The CoinGecko Demo plan has a rate limit. If we accidentally exceed it,
      the API returns HTTP 429 (Too Many Requests). Without retries, the whole
      pipeline would fail and we'd need to restart manually. With retries +
      exponential-style backoff, we recover automatically.
 
    Args:
        endpoint : API path to append to BASE_URL (e.g., "/coins/markets")
        params   : Query parameters dict (API key is injected automatically)
 
    Returns:
        Parsed JSON response (dict or list depending on endpoint)
 
    Raises:
        requests.HTTPError if all retries are exhausted
    """
    if params is None:
        params = {}
 
    # Always inject the API key — never inline it in the URL string
    params["x_cg_demo_api_key"] = API_KEY
 
    url = f"{BASE_URL}{endpoint}"
 
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.debug(f"Calling {url} | attempt {attempt}")
            response = requests.get(url, params=params, timeout=30)
 
            # raise_for_status() throws an exception for 4xx/5xx HTTP codes.
            # This is safer than checking response.ok manually.
            response.raise_for_status()
 
            return response.json()
 
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response else "unknown"
 
            if status == 429:
                # Rate limit hit: wait longer before retrying
                wait = RETRY_BACKOFF_SEC * attempt * 2
                logger.warning(
                    f"Rate limit hit (429) on {endpoint}. "
                    f"Waiting {wait}s before retry {attempt}/{MAX_RETRIES}"
                )
                time.sleep(wait)
            elif attempt == MAX_RETRIES:
                logger.error(f"All {MAX_RETRIES} retries failed for {endpoint}: {e}")
                raise
            else:
                logger.warning(f"HTTP {status} on attempt {attempt} for {endpoint}: {e}")
                time.sleep(RETRY_BACKOFF_SEC)
 
        except requests.exceptions.Timeout:
            logger.warning(f"Timeout on attempt {attempt} for {endpoint}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF_SEC)
 
        except requests.exceptions.ConnectionError as e:
            logger.warning(f"Connection error on attempt {attempt} for {endpoint}: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF_SEC)
 
 
def build_envelope(data: Any, source_endpoint: str, extra_meta: Optional[Dict] = None) -> Dict:
    """
    Wrap the raw API payload in a standard metadata envelope.
 
    WHY AN ENVELOPE?
      When Autoloader later reads these JSON files into Bronze Delta tables,
      it needs to know:
        - WHEN was this data collected?      → ingestion_timestamp
        - WHICH pipeline run produced it?   → pipeline_run_id
        - WHERE did the data come from?     → source_endpoint
      These three fields provide full data lineage from Bronze all the way
      back to the source API call. Without them, debugging bad data is
      nearly impossible.
 
    Args:
        data            : The raw API response (dict or list)
        source_endpoint : Human-readable endpoint label (e.g., "coins_markets")
        extra_meta      : Optional dict of additional metadata (e.g., {"coin_id": "bitcoin"})
 
    Returns:
        Dict with standard envelope fields + the original payload under "payload"
    """
    envelope = {
        "ingestion_timestamp" : NOW_UTC.isoformat(),      # ISO 8601, always UTC
        "pipeline_run_id"     : RUN_ID,
        "source_endpoint"     : source_endpoint,
        "top_n_coins_config"  : TOP_N_COINS,              # Record the config used
        "schema_version"      : "1.0",                    # For future schema migrations
        "payload"             : data,
    }
 
    if extra_meta:
        envelope.update(extra_meta)
 
    return envelope
 
 
def save_to_landing(envelope: Dict, subfolder: str, filename: str) -> str:
    """
    Serialize envelope to JSON and write to ADLS landing container.
 
    WHY DBUTILS.FS.PUT?
      dbutils.fs is Databricks' file system abstraction that transparently
      handles authentication to ADLS Gen2. Under the hood it uses the
      OAuth2 token that Databricks manages for you, so you never deal with
      connection strings or SAS tokens in notebook code.
 
    WHY DATE-PARTITIONED PATHS?
      Organizing files as /YYYY/MM/DD/ means:
        - Autoloader can filter by date (process only today's files)
        - Analysts can quickly find files for a specific date
        - Old files are easy to archive/delete by month
        - Azure Data Lake's lifecycle policies can auto-tier old data to cold storage
 
    Args:
        envelope  : The full envelope dict to serialize
        subfolder : Path under landing container (e.g., "coins_markets")
        filename  : File name (e.g., "coins_markets_20241115_060000.json")
 
    Returns:
        Full ADLS path where the file was written
    """
    # Construct the full path: abfss://landing@<account>.dfs.core.windows.net/<subfolder>/YYYY/MM/DD/<filename>
    full_path = f"{ADLS_ROOT}/{subfolder}/{DATE_PATH}/{filename}"
 
    # json.dumps with default=str handles non-serialisable types (e.g., datetime objects)
    # indent=2 makes the file human-readable when you open it in Azure Storage Explorer
    json_content = json.dumps(envelope, indent=2, default=str)
 
    # overwrite=True: if this exact path already exists (e.g., re-run same minute),
    # replace it. For the landing layer this is acceptable since the timestamp
    # in the filename makes collisions extremely unlikely in normal operation.
    dbutils.fs.put(full_path, json_content, overwrite=True)
 
    logger.info(f"✓ Saved → {full_path}")
    return full_path

In [0]:
# -----------------------------------------------------------------------------
# CELL 5: ENDPOINT 1 — /coins/markets
#
# WHAT: Fetches a live market snapshot for the top TOP_N_COINS coins,
#       ranked by market capitalisation, priced in USD.
#
# WHY FIRST: This endpoint gives us the list of coin IDs we need for all
#   subsequent per-coin calls (OHLC and market_chart). We extract coin_ids
#   from this response and reuse them in the loops below.
#
# OUTPUT: 1 JSON file per daily run
# -----------------------------------------------------------------------------
 
logger.info("─" * 60)
logger.info("ENDPOINT 1: /coins/markets — top market snapshot")
 
markets_data = call_api(
    "/coins/markets",
    params={
        "vs_currency" : "usd",            # Express all prices in US dollars
        "order"       : "market_cap_desc", # Biggest coins first
        "per_page"    : TOP_N_COINS,       # Number of coins to return
        "page"        : 1,                 # First (and only) page
        # Include 7-day price change sparkline data for richer analysis
        "sparkline"   : "false",           # Set true if you want sparkline arrays (larger payload)
    }
)
 
# Save to landing
markets_path = save_to_landing(
    envelope  = build_envelope(markets_data, "coins_markets"),
    subfolder = "coins_markets",
    filename  = f"coins_markets_{TS_SUFFIX}.json"
)
 
# Extract coin IDs — these drive all the per-coin loops below
coin_ids: List[str] = [coin["id"] for coin in markets_data]
logger.info(f"  Collected {len(coin_ids)} coins: {coin_ids[:5]} ... (truncated)")
 
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 6: ENDPOINTS 2 & 3 — /coins/{id}/ohlc and /coins/{id}/market_chart
#
# WHAT:
#   For EACH of the TOP_N_COINS coins we make TWO API calls:
#     - /ohlc          → 90 days of OHLC (candlestick) data
#     - /market_chart  → 30 days of hourly price, market cap, and volume data
#
# WHY LOOP BOTH TOGETHER?
#   Doing both calls for one coin before moving to the next coin means:
#   - The sleep between calls is naturally distributed
#   - If the pipeline fails mid-way, you know exactly which coin caused it
#   - Easier to implement per-coin checkpointing in future
#
# RATE LIMIT STRATEGY:
#   Demo plan = ~30 calls/min. We sleep 1.2s between each coin.
#   50 coins × 2 endpoints = 100 calls.
#   100 calls × 1.2s = 120s = ~2 minutes total for this loop.
#   This keeps us well under the rate limit.
#
# OUTPUT: 2 JSON files per coin = 100 files total at TOP_N_COINS=50
# -----------------------------------------------------------------------------
 
logger.info("─" * 60)
logger.info(f"ENDPOINTS 2 & 3: OHLC + Market Chart for {len(coin_ids)} coins")
 
ohlc_paths         = []
market_chart_paths = []
 
for index, coin_id in enumerate(coin_ids, start=1):
    logger.info(f"  [{index:>2}/{len(coin_ids)}] Processing coin: {coin_id}")
 
    # ── Endpoint 2: OHLC ─────────────────────────────────────────────────────
    # days=90 gives us 90 days of OHLC candles.
    # On the Demo plan, data comes in daily intervals for 31-90 day ranges.
    # Each candle = [timestamp_ms, open, high, low, close]
    try:
        ohlc_data = call_api(
            f"/coins/{coin_id}/ohlc",
            params={"vs_currency": "usd", "days": 90}
        )
        ohlc_path = save_to_landing(
            envelope  = build_envelope(
                ohlc_data,
                source_endpoint = "ohlc",
                extra_meta      = {"coin_id": coin_id}   # Tag which coin this is
            ),
            subfolder = f"ohlc/{coin_id}",
            filename  = f"ohlc_{coin_id}_{TS_SUFFIX}.json"
        )
        ohlc_paths.append(ohlc_path)
    except Exception as e:
        # Log and continue — one failed coin should not stop the whole pipeline
        logger.error(f"  OHLC failed for {coin_id}: {e}")
 
    # ── Endpoint 3: Market Chart ──────────────────────────────────────────────
    # days=30 gives hourly data points for the last 30 days.
    # Response has three parallel arrays: prices, market_caps, total_volumes
    # Each entry is [timestamp_ms, value]
    try:
        chart_data = call_api(
            f"/coins/{coin_id}/market_chart",
            params={"vs_currency": "usd", "days": 30}
        )
        chart_path = save_to_landing(
            envelope  = build_envelope(
                chart_data,
                source_endpoint = "market_chart",
                extra_meta      = {"coin_id": coin_id}
            ),
            subfolder = f"market_chart/{coin_id}",
            filename  = f"market_chart_{coin_id}_{TS_SUFFIX}.json"
        )
        market_chart_paths.append(chart_path)
    except Exception as e:
        logger.error(f"  market_chart failed for {coin_id}: {e}")
 
    # Sleep AFTER both calls for this coin before moving to the next
    # This gives a natural cadence of ~2 API calls per 1.2 seconds
    if index < len(coin_ids):   # No need to sleep after the last coin
        time.sleep(RATE_LIMIT_SLEEP)
 
logger.info(f"  Saved {len(ohlc_paths)} OHLC files")
logger.info(f"  Saved {len(market_chart_paths)} market_chart files")
 
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 7: ENDPOINT 4 — /search/trending
#
# WHAT: Returns the top 7 coins trending on CoinGecko based on search traffic.
#
# WHY: This is a sentiment/social signal. A coin trending on search traffic
#   often precedes a price move. We capture this daily snapshot to later join
#   with market data to see if trending coins also have unusual volume.
#
# OUTPUT: 1 JSON file per daily run
# -----------------------------------------------------------------------------
 
logger.info("─" * 60)
logger.info("ENDPOINT 4: /search/trending — trending coins")
 
trending_data = call_api("/search/trending")
 
trending_path = save_to_landing(
    envelope  = build_envelope(trending_data, "trending"),
    subfolder = "trending",
    filename  = f"trending_{TS_SUFFIX}.json"
)
 
logger.info(f"  Trending coins: {[c['item']['id'] for c in trending_data.get('coins', [])]}")

In [0]:
# -----------------------------------------------------------------------------
# CELL 8: ENDPOINT 5 — /global
#
# WHAT: Returns a single row of global crypto market statistics.
#   Includes total market cap, BTC dominance, ETH dominance, active coins.
#
# WHY: This gives macro context for the coin-level data. Knowing that BTC
#   dominance dropped 2% while a coin surged helps analysts distinguish
#   a coin-specific move from a market-wide rotation.
#
# OUTPUT: 1 JSON file per daily run
# -----------------------------------------------------------------------------
 
logger.info("─" * 60)
logger.info("ENDPOINT 5: /global — global market statistics")
 
global_data = call_api("/global")
 
global_path = save_to_landing(
    envelope  = build_envelope(global_data, "global"),
    subfolder = "global",
    filename  = f"global_{TS_SUFFIX}.json"
)
 
btc_dom = global_data.get("data", {}).get("market_cap_percentage", {}).get("btc", "N/A")
logger.info(f"  BTC Dominance: {btc_dom:.2f}%" if isinstance(btc_dom, float) else f"  BTC Dominance: {btc_dom}")

In [0]:
# -----------------------------------------------------------------------------
# CELL 9: INGESTION SUMMARY
# Emit a structured summary that can be parsed by Azure Monitor / Log Analytics
# -----------------------------------------------------------------------------
 
summary = {
    "event"               : "landing_ingestion_complete",
    "pipeline_run_id"     : RUN_ID,
    "run_timestamp_utc"   : NOW_UTC.isoformat(),
    "coins_collected"     : len(coin_ids),
    "ohlc_files_saved"    : len(ohlc_paths),
    "market_chart_files"  : len(market_chart_paths),
    "trending_saved"      : True,
    "global_saved"        : True,
    "top_n_coins_config"  : TOP_N_COINS,
    "status"              : "SUCCESS"
}
 
logger.info("=" * 70)
logger.info("LANDING INGESTION COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<28}: {v}")
logger.info("=" * 70)
 
 